# 01 Synthetic Dynamics

## Purpose

Compare logistic-map, AR(1), and GARCH benchmark processes using package code only.

## Data / config used

- synthetic benchmark systems
- output directory: `outputs/notebooks/01_synthetic_dynamics/`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from dynsys_econometrics.extremes import threshold_exceedances
from dynsys_econometrics.plots import plot_clustered_vs_isolated_extremes
from dynsys_econometrics.simulation import simulate_ar1, simulate_garch11, simulate_logistic_map

output_dir = Path("outputs/notebooks/01_synthetic_dynamics")
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
logistic = simulate_logistic_map(n=1200, r=4.0, x0=0.17)
logistic["value"] = -np.log(np.abs(logistic["value"] - 0.5) + 1e-12)

ar1 = pd.DataFrame(
    {
        "date": logistic["date"],
        "series_id": "ar1",
        "value": np.abs(simulate_ar1(n_steps=1200, phi=0.8, sigma=1.0, seed=7)),
    }
)

garch = pd.DataFrame(
    {
        "date": logistic["date"],
        "series_id": "garch11",
        "value": np.abs(simulate_garch11(n_steps=1200, seed=11)),
    }
)

panel = pd.concat([logistic[["date", "series_id", "value"]], ar1, garch], ignore_index=True)
events = threshold_exceedances(panel, quantile=0.95)
events.to_csv(output_dir / "synthetic_events.csv", index=False)

plot_clustered_vs_isolated_extremes(
    clustered_series=garch.set_index("date")["value"].iloc[:400],
    isolated_series=ar1.set_index("date")["value"].iloc[:400],
    clustered_threshold=float(garch["value"].quantile(0.97)),
    isolated_threshold=float(ar1["value"].quantile(0.97)),
    output_path=output_dir / "synthetic_extremes_comparison.png",
)

events.groupby("series_id")["exceedance"].sum().to_frame("n_exceedances")


## Limitations

These benchmark systems are used to compare recurrence and clustering behavior. They are not realistic models of a macroeconomy.
